In [1]:
import pandas as pd

# pandas reads the CSV correctly even when text fields contain line breaks
listings_csv = pd.read_csv("../data/raw/nyc/listings.csv", low_memory=False)
print("Listings rows in CSV (pandas):", len(listings_csv))

Listings rows in CSV (pandas): 30259


In [2]:
# See which columns the calendar actually has
import duckdb

CAL = "../data/bronze/nyc/calendar.parquet"
LST = "../data/bronze/nyc/listings.parquet"


duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{CAL}')").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ listing_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ available      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ minimum_nights │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ maximum_nights │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [3]:
# Investigate the extra calendar rows

duckdb.sql(f"""
    SELECT
        COUNT(*)                     AS total_rows,
        COUNT(DISTINCT listing_id)   AS distinct_listings,
        MIN(date)                    AS first_date,
        MAX(date)                    AS last_date
    FROM read_parquet('{CAL}')
""").show()


┌────────────┬───────────────────┬────────────┬────────────┐
│ total_rows │ distinct_listings │ first_date │ last_date  │
│   int64    │       int64       │  varchar   │  varchar   │
├────────────┼───────────────────┼────────────┼────────────┤
│   11152576 │             30555 │ 2026-06-14 │ 2027-06-22 │
└────────────┴───────────────────┴────────────┴────────────┘



In [4]:
# How many dates does each listing have?
duckdb.sql(f"""
    SELECT days_per_listing, COUNT(*) AS number_of_listings
    FROM (
        SELECT listing_id, COUNT(*) AS days_per_listing
        FROM read_parquet('{CAL}')
        GROUP BY listing_id
    )
    GROUP BY days_per_listing
    ORDER BY number_of_listings DESC
    LIMIT 10
""").show()

┌──────────────────┬────────────────────┐
│ days_per_listing │ number_of_listings │
│      int64       │       int64        │
├──────────────────┼────────────────────┤
│              365 │              30554 │
│              366 │                  1 │
└──────────────────┴────────────────────┘



In [5]:
# Duplicates and unmatched listings
duckdb.sql(f"""
    SELECT
        -- same listing + same date appearing more than once
        (SELECT COUNT(*) FROM (
            SELECT listing_id, date
            FROM read_parquet('{CAL}')
            GROUP BY listing_id, date
            HAVING COUNT(*) > 1
        )) AS duplicate_listing_dates,

        -- calendar listings that do not exist in listings
        (SELECT COUNT(DISTINCT c.listing_id)
         FROM read_parquet('{CAL}') c
         LEFT JOIN read_parquet('{LST}') l ON c.listing_id = l.id
         WHERE l.id IS NULL) AS calendar_listings_not_in_listings
""").show()

┌─────────────────────────┬───────────────────────────────────┐
│ duplicate_listing_dates │ calendar_listings_not_in_listings │
│          int64          │               int64               │
├─────────────────────────┼───────────────────────────────────┤
│                       0 │                               296 │
└─────────────────────────┴───────────────────────────────────┘



In [6]:
# How complete is the listings price?
duckdb.sql(f"""
    SELECT
        COUNT(*)                                        AS total_listings,
        COUNT(price)                                    AS listings_with_price,
        ROUND(100.0 * COUNT(price) / COUNT(*), 1)       AS pct_with_price,
        MEDIAN(TRY_CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DOUBLE))
                                                        AS median_price
    FROM read_parquet('{LST}')
""").show()

┌────────────────┬─────────────────────┬────────────────┬──────────────┐
│ total_listings │ listings_with_price │ pct_with_price │ median_price │
│     int64      │        int64        │     double     │    double    │
├────────────────┼─────────────────────┼────────────────┼──────────────┤
│          30259 │               21515 │           71.1 │       174.69 │
└────────────────┴─────────────────────┴────────────────┴──────────────┘



In [7]:
# Do listings start on different dates?
duckdb.sql(f"""
    SELECT first_date, COUNT(*) AS number_of_listings
    FROM (
        SELECT listing_id, MIN(date) AS first_date
        FROM read_parquet('{CAL}')
        GROUP BY listing_id
    )
    GROUP BY first_date
    ORDER BY first_date
""").show()

┌────────────┬────────────────────┐
│ first_date │ number_of_listings │
│  varchar   │       int64        │
├────────────┼────────────────────┤
│ 2026-06-14 │              14544 │
│ 2026-06-15 │               7224 │
│ 2026-06-22 │               2088 │
│ 2026-06-23 │               6699 │
└────────────┴────────────────────┘



In [8]:
# Does every non-empty price convert to a number?
duckdb.sql(f"""
    SELECT
        COUNT(price) AS non_empty_price,
        COUNT(TRY_CAST(REPLACE(REPLACE(price, '$', ''), ',', '') AS DOUBLE))
                     AS price_converted_ok
    FROM read_parquet('{LST}')
""").show()

┌─────────────────┬────────────────────┐
│ non_empty_price │ price_converted_ok │
│      int64      │       int64        │
├─────────────────┼────────────────────┤
│           21515 │              21515 │
└─────────────────┴────────────────────┘

